# 09 — findings detector: held-out test + locked-50 sanity

Scores `findings_best.pt` on the DentalOPG **test** split (154 images, never
used for training or model selection), then runs it on the locked DENTEX 50 as
a domain-shift sanity check. The locked 50 has no missing/BDR/filling ground
truth, so that section reports detection counts only — no accuracy claim.


In [ ]:
import json
import shutil
from pathlib import Path

import yaml
from ultralytics import YOLO

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = Path("/tmp/opencode/dentalopg")
OUT = Path("/tmp/opencode/findings_test")
MODEL = Path("/tmp/opencode/kaggle_out/findings/findings_best.pt")
assert MODEL.exists(), f"run notebooks/fetch_findings.sh first: {MODEL}"

MAP = {0: 0, 1: 1, 2: 2, 3: None, 4: 3, 5: 4, 6: 5, 7: 6,
       8: None, 9: None, 10: 7, 11: 8, 12: 9, 13: 10}
NAMES = ["BDR", "Bridge", "Cap", "Distoangular Impaction", "Filling",
         "Horizontal Impaction", "Missing Teeth", "Periodontitis",
         "Root Canal", "Unerupted Wisdom", "Apical Abscess"]

for sub in ("images", "labels"):
    (OUT / sub).mkdir(parents=True, exist_ok=True)
for img in sorted((SRC / "test" / "images").iterdir()):
    lab = SRC / "test" / "labels" / (img.stem + ".txt")
    if not lab.exists():
        continue
    rows = []
    for line in lab.read_text().splitlines():
        parts = line.split()
        if len(parts) != 5:
            continue
        new = MAP[int(parts[0])]
        if new is None:
            continue
        rows.append(f"{new} " + " ".join(parts[1:]))
    shutil.copy2(img, OUT / "images" / img.name)
    (OUT / "labels" / (img.stem + ".txt")).write_text(
        "\n".join(rows) + ("\n" if rows else ""))

(OUT / "test.txt").write_text(
    "\n".join(str(p) for p in sorted((OUT / "images").iterdir())) + "\n")
(OUT / "data.yaml").write_text(yaml.safe_dump(
    {"test": str(OUT / "test.txt"),
     "names": {i: n for i, n in enumerate(NAMES)}}, sort_keys=False))
print("test images:", len(list((OUT / "images").iterdir())))


In [ ]:
model = YOLO(str(MODEL))
m = model.val(data=str(OUT / "data.yaml"), split="test", imgsz=640, conf=0.001, iou=0.6)
names = [m.names[i] for i in sorted(m.names)]
ap50 = m.box.ap50.tolist()

print(f"mAP50    {m.box.map50:.4f}")
print(f"mAP50-95 {m.box.map:.4f}")
print(f"precision {m.box.mp:.4f} | recall {m.box.mr:.4f}")
print()
for name, ap in sorted(zip(names, ap50), key=lambda t: -t[1]):
    print(f"  {name:24s} AP50 {ap:.3f}")

summary = {
    "split": "DentalOPG test (154 images, held out)",
    "model": str(MODEL), "imgsz": 640,
    "mAP50": m.box.map50, "mAP50_95": m.box.map,
    "precision": m.box.mp, "recall": m.box.mr,
    "ap50_per_class": dict(zip(names, ap50)), "classes": NAMES,
}
out = REPO / "dentex_week3/eval/findings_test_metrics.json"
out.write_text(json.dumps(summary, indent=2))
print("wrote", out)


## Locked-50 sanity check (domain shift)

DENTEX panoramas are ~2,900x1,500; DentalOPG images are 640x640 letterboxed
from a different hospital population. Counts only — there is no findings GT on
the locked 50.


In [ ]:
from collections import Counter

XRAYS = Path("/home/sayfeldin/Projects/dental-model-research/data/raw/dentex"
             "/validation_data/quadrant_enumeration_disease/xrays")
counts = Counter()
per_image = {}
for img in sorted(XRAYS.glob("*.png")):
    res = model.predict(str(img), imgsz=640, conf=0.25, verbose=False)[0]
    found = [NAMES[int(c)] for c in res.boxes.cls.tolist()]
    per_image[img.name] = found
    counts.update(found)

print("images:", len(per_image), "| detections:", sum(counts.values()))
for name, n in counts.most_common():
    print(f"  {name:24s} {n}")
(Path("/tmp/opencode/findings_locked50.json")).write_text(
    json.dumps(per_image, indent=2))
